<sub>Copyright (c) 2026 Yassine Moukil ELECTRO-GAMMA. All rights reserved. &middot; Author: Yassine Moukil ELECTRO-GAMMA</sub>


# Build AndroProgX APK (Google Colab)

Run each cell top to bottom (or **Runtime → Run all**). Total time is usually 5–10 minutes, mostly downloads.

**Before you start:** nothing to upload — step 1 clones the project straight from GitHub.

**What you get:** a **signed release APK** (the one to install and keep), a debug APK, and the **keystore** the release APK was signed with. The release build is only signed when the `ANDROPROGX_KEYSTORE` variables from step 7 are set (see `app/build.gradle`); without a key, Android refuses to install a release APK, so step 7 matters.

**Tip:** if you open this notebook in your **phone's browser** (colab.research.google.com works on mobile), the final `files.download()` step will save the APKs straight to your phone's Downloads folder, so you can install with no extra transfer step.

In [ ]:
# 1. Get the project from GitHub (no manual upload needed)
# The repo owner/name below is filled in automatically by github_sync.py when you push.
GITHUB_USER = "Electro-Gamma"
GITHUB_REPO = "AndroProgX"
BRANCH = "main"

!rm -rf /content/AndroProgX
!git clone --depth 1 -b {BRANCH} https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git /content/AndroProgX

# Fallback: if the repo is private or you prefer a local copy, comment the clone above and use:
# from google.colab import files
# files.upload()   # select AndroProgX.zip, then unzip it in the next cell

In [ ]:
# 2. Check the project is in place
!ls /content/AndroProgX
# (Zip fallback only:  !unzip -q -o AndroProgX.zip -d /content/ )

In [ ]:
# 3. Install JDK 17 (required by the project's compileOptions)
import os
!apt-get update -qq
!apt-get install -y -qq openjdk-17-jdk
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = "/usr/lib/jvm/java-17-openjdk-amd64/bin:" + os.environ["PATH"]
!java -version

In [ ]:
# 4. Download the Android SDK command-line tools and lay them out correctly
import os
os.environ["ANDROID_HOME"] = "/content/android-sdk"
os.environ["ANDROID_SDK_ROOT"] = "/content/android-sdk"

!mkdir -p /content/android-sdk/cmdline-tools
!wget -q https://dl.google.com/android/repository/commandlinetools-linux-14742923_latest.zip -O /content/cmdline-tools.zip
!unzip -q /content/cmdline-tools.zip -d /content/android-sdk/cmdline-tools
!mv /content/android-sdk/cmdline-tools/cmdline-tools /content/android-sdk/cmdline-tools/latest

os.environ["PATH"] += ":/content/android-sdk/cmdline-tools/latest/bin:/content/android-sdk/platform-tools"
!mkdir -p ~/.android && touch ~/.android/repositories.cfg
!sdkmanager --version

In [ ]:
# 5. Accept licenses and install the platform + build-tools the project needs
# (If build-tools 35.0.0 is ever unavailable, run `!sdkmanager --list` to see current versions
#  and swap the version string below.)
!yes | sdkmanager --licenses > /dev/null
!sdkmanager "platform-tools" "platforms;android-35" "build-tools;35.0.0"

with open("/content/AndroProgX/local.properties", "w") as f:
    f.write("sdk.dir=/content/android-sdk\n")

In [ ]:
# 6. Download Gradle 8.7 (matches this project's configuration)
import os
!wget -q https://services.gradle.org/distributions/gradle-8.7-bin.zip -O /content/gradle-8.7-bin.zip
!unzip -q /content/gradle-8.7-bin.zip -d /content/
os.environ["PATH"] += ":/content/gradle-8.7/bin"
!gradle -v

In [ ]:
# 7. Generate a signing key for the release build
#
# app/build.gradle only signs the release build when ANDROPROGX_KEYSTORE points at a real
# file - without one, assembleRelease produces an UNSIGNED apk and Android refuses to
# install it. This generates a fresh, self-signed key with keytool (bundled with the JDK
# from step 3) so the release build installs.
#
# Store password and key password are deliberately the SAME value: keytool's
# default format since JDK 9 is PKCS12, which doesn't support the two differing -
# given a different -keypass it silently ignores it and encrypts the key with the
# store password anyway, so a genuinely different key password only makes Gradle
# fail later with a BadPaddingException. One shared password sidesteps that.
#
# An APK signed with a different key installs fine on its own, but Android won't
# let it upgrade-in-place over an existing install signed with another key (the
# debug key, or an earlier throwaway key) - uninstall the old one first if you
# hit that. To make future builds upgrade over this one, keep the keystore file
# and the password printed below, and reuse them instead of generating a new key.
import os, secrets

os.environ["ANDROPROGX_KEYSTORE"] = "/content/androprogx-release.keystore"
_password = secrets.token_urlsafe(24)
os.environ["ANDROPROGX_KEYSTORE_PASSWORD"] = _password
os.environ["ANDROPROGX_KEY_ALIAS"] = "androprogx"
os.environ["ANDROPROGX_KEY_PASSWORD"] = _password

!rm -f "{os.environ['ANDROPROGX_KEYSTORE']}"
!keytool -genkeypair \
    -keystore "{os.environ['ANDROPROGX_KEYSTORE']}" \
    -alias "{os.environ['ANDROPROGX_KEY_ALIAS']}" \
    -keyalg RSA -keysize 2048 -validity 10000 \
    -storepass "{os.environ['ANDROPROGX_KEYSTORE_PASSWORD']}" \
    -keypass "{os.environ['ANDROPROGX_KEY_PASSWORD']}" \
    -dname "CN=AndroProgX Personal Build, OU=Personal, O=Personal, L=Unknown, ST=Unknown, C=US"

assert os.path.exists(os.environ["ANDROPROGX_KEYSTORE"]), "keytool didn't produce a keystore file"
print("Keystore ready. To let a future rebuild upgrade over this install instead of")
print("needing an uninstall, keep the keystore file and reuse this password:")
print("  alias:   ", os.environ["ANDROPROGX_KEY_ALIAS"])
print("  password:", _password)

In [ ]:
# 8. Build the debug and release APKs (this is the step that takes the longest -- the
#    first Gradle sync downloads all dependencies, including usb-serial-for-android via
#    JitPack). The signing variables from step 7 are inherited by Gradle, which is what
#    signs the release APK with your key.
%cd /content/AndroProgX
!gradle assembleDebug assembleRelease --stacktrace

In [ ]:
# 9. Verify the release signature, then download both APKs and the keystore from step 7
#
# This project names EVERY build AndroProgX.apk (see the applicationVariants block in
# app/build.gradle); they only differ by folder. They are copied to distinct names first,
# otherwise the browser would save the second one as "AndroProgX (1).apk".
import glob, os, shutil, subprocess
from google.colab import files

debug_matches = glob.glob("/content/AndroProgX/app/build/outputs/apk/debug/*.apk")
release_matches = glob.glob("/content/AndroProgX/app/build/outputs/apk/release/*.apk")
print("Debug APK:", debug_matches)
print("Release APK:", release_matches)
assert release_matches, "No release APK found - scroll up to the build cell to check for errors"
assert debug_matches, "No debug APK found - scroll up to the build cell to check for errors"

# The file name cannot tell a signed release APK from an unsigned one (both are AndroProgX.apk),
# so check the signature itself: it must verify and carry the key generated in step 7.
apksigner = "/content/android-sdk/build-tools/35.0.0/apksigner"
check = subprocess.run([apksigner, "verify", "--print-certs", release_matches[0]], capture_output=True, text=True)
print(check.stdout.strip() or check.stderr.strip())
assert check.returncode == 0, (
    "The release APK is unsigned or invalid - run step 7 (signing key) in this same session before building")
assert "AndroProgX Personal Build" in check.stdout, "The release APK is not signed with the step-7 key"

release_apk = "/content/AndroProgX-release.apk"
debug_apk = "/content/AndroProgX-debug.apk"
shutil.copy(release_matches[0], release_apk)
shutil.copy(debug_matches[0], debug_apk)

files.download(release_apk)
files.download(debug_apk)
files.download(os.environ["ANDROPROGX_KEYSTORE"])  # keep it (with the password from step 7) to upgrade over this install later

## Installing on your phone

- **Opened this notebook on your phone?** The APKs just landed in your Downloads folder — tap `AndroProgX-release.apk` (the one to keep), allow "install from this source" if prompted, and install. The debug APK is there if you ever need to attach a debugger.
- **Built it on a PC?** Copy the `.apk` to your phone (USB cable, Drive, email, etc.), then tap it to install, or run `adb install AndroProgX-release.apk` with the phone connected via USB debugging.
- The release APK is signed with the throwaway keystore step 7 generated. That is fine for installing fresh, but Android will refuse to upgrade an existing install signed with a *different* key (the debug build, or a previous build signed with another throwaway key) — uninstall the old one first if that happens. To make future builds upgrade over this one, keep the downloaded keystore and the password printed in step 7.
- The debug APK is self-signed with Android's default debug key, which is fine for sideloading on your own device.

## If the build fails

Scroll up to the "Build the debug and release APKs" cell's output — Gradle's `--stacktrace` error points at the actual problem. Paste the `e:` / `error:` lines (not the whole log) back and it's a quick fix. If the download cell says the release APK is unsigned, step 7 wasn't run in this session (the keystore variables only live as long as the Colab runtime).